<div style="text-align: center;">
    <img src="./assets/ki_ag_main.jpg" alt="Logo" style="width: 1000px;"/>
</div>

<center><h1>KI-AG Workshop - n8n</h1></center>

<center>
    
</center>

In diesem Notebook lernst du, wie du deine trainierten LoRA-Adapter in Ollama integrierst und sie anschließend in einem n8n-Workflow verwendest, um komplexe KI-gestützte Szenarien zu simulieren.



# Kapitel 1: Einrichten der Umgebung

## 1.1 Arbeitsverzeichnis öffnen
Navigiere im Windows Explorer zu folgendem Pfad:
> `C:\Tools\3_n8n`

## 1.2 Kommandozeile starten
Öffne die Eingabeaufforderung (cmd) direkt in diesem Ordner:
1. Klicke oben in die **Adresszeile** des Explorers.
2. Tippe `cmd` ein.
3. Drücke `Enter`.

Es öffnet sich nun ein Terminal-Fenster im passenden Pfad.

## 1.3 Python-Environment aktivieren
Aktiviere nun das dort hinterlegte Python-Environment.

```cmd
.venv\Scripts\activate
```

**Sobald das erledigt ist, bist du bereit!**

## 1.4 N8n starten
Starte n8n in einem **separaten** cmd-Fenster.
Gib dazu einfach folgenden Befehl ein:
```cmd
n8n
```

Falls ein Login-Screen erscheint, schließe n8n im gleichen Terminal-Fenster wieder mit `Strg+C`.
Setze anschließend das User-Management mit folgendem Befehl zurück:
```cmd
n8n user-management:reset
```
Danach kannst du n8n erneut starten.

Wenn n8n neu gestartet ist, erscheint eine Registrierungsmaske im Browser.
Fülle diese aus, um fortzufahren.

> **Hinweis:** Die Login-Daten werden nur lokal auf diesem Rechner gespeichert und sind nur temporär gültig bis zum nächsten Reset.

# Kapitel 2: Personas in Ollama

In diesem Kapitel lernst du, wie du deine trainierten LoRA-Adapter in Ollama integrierst, um sie lokal und effizient nutzen zu können.
Alle notwendigen Skripte und Daten befinden sich bereits im Ordner `C:\Tools\3_n8n`.

## 2.1 GGUF Konvertierung

### Was ist GGUF?
GGUF (GPT-Generated Unified Format) ist ein binäres Dateiformat, das speziell für die schnelle und effiziente Inferenz von Sprachmodellen auf CPUs und GPUs (insbesondere mit Apple Silicon oder Consumer-Grafikkarten) entwickelt wurde. Es ermöglicht, Modelle kompakt zu speichern und direkt mit Tools wie Ollama oder llama.cpp zu laden.

### Skripte
Wir nutzen zwei Skripte für die Umwandlung:
1. `convert_hf_to_gguf.py`: Konvertiert Basis-Modelle (Hugging Face Format) nach GGUF.
2. `convert_lora_to_gguf.py`: Konvertiert LoRA-Adapter nach GGUF.

### Anleitung
Führe die Konvertierung in deiner **aktivierten Python-Umgebung** (siehe Kapitel 1) aus.

Da wir LoRA-Adapter nutzen, ist vor allem das zweite Skript wichtig. Es ist interaktiv gestaltet.
Führe folgenden Befehl aus:

```cmd
python convert_lora_to_gguf.py
```

Folge den Anweisungen im Terminal, um deine Adapter (z.B. `ac-host`, `angry-customer`, `scorer`) in das GGUF-Format umzuwandeln. Die Ergebnisse landen im Ordner `gguf_models`.

## 2.2 Modelfiles anpassen

### Was sind Modelfiles?
Ein **Modelfile** ist die Konfigurationsdatei für Ollama. Es definiert, welches Basis-Modell verwendet wird, welcher Adapter geladen werden soll, und setzt Parameter wie System-Prompt, Temperatur oder Stop-Sequenzen.

### Anleitung
Im Ordner `3_n8n_with_lora` findest du bereits vorbereitete Modelfiles:
- `Modelfile.ac-host-extended`
- `Modelfile.angry-customer-extended`
- `Modelfile.scorer-extended`

Öffne diese Dateien mit einem Texteditor (z.B. VS Code oder Notepad) und passe die Pfade an.
Stelle sicher, dass die Zeilen `FROM` und `ADAPTER` auf die korrekten, soeben erstellten GGUF-Dateien zeigen.
Stelle auch sicher, dass der Systemprompt der exakte ist wie in den Trainingsdateien.

**Beispiel:**
```dockerfile
FROM ./gguf_models/base/meta-llama_Llama-3.1-8B-Instruct.gguf
ADAPTER ./gguf_models/persona/ac-host-extended.gguf
```

## 2.3 Ollama einrichten und starten

### 1. Server starten
Starte den Ollama-Server in einem **separaten** CMD-Fenster:
```cmd
ollama serve
```

### 2. Modelle prüfen
Öffne ein **weiteres** CMD-Fenster und prüfe, welche Modelle bereits existieren:
```cmd
ollama list
```

### 3. Modelle erstellen
Erstelle nun die drei Persona-Modelle basierend auf deinen angepassten Modelfiles.
Navigiere dazu in das Verzeichnis `C:\Tools\3_n8n\2_fine_tuning\3_n8n_with_lora`.

Verwende folgendes Schema, um die Modelle zu erstellen:
```cmd
ollama create [Modellname] -f [Modelfile-Pfad]
```

Führe dies für alle drei Modelle (`ac-host`, `angry-customer`, `scorer`) mit den entsprechenden Modelfiles aus.

### 4. Modelle testen
Du kannst die Modelle nun direkt im Terminal testen:
```cmd
ollama run [Modellname]
```

---

**Hinweis:**
Du kannst auf genau demselben Weg auch deine **eigenen LoRA-Adapter**, die du im vorherigen Workshop-Teil erstellt hast, in Ollama einbinden!

# Kapitel 3: n8n

In diesem Kapitel importieren wir den vorbereiteten Workflow in n8n, um unsere Persona-Modelle in einer simulierten Umgebung zu nutzen.

## 3.1 n8n öffnen
Stelle sicher, dass n8n läuft (siehe Kapitel 1.4).
Öffne anschließend folgende Adresse in deinem Browser:

[http://localhost:5678](http://localhost:5678)

## 3.2 Beschreibung der Umgebung

- am rechten oberen Rand das **+** öffnet das **Node Panel**, lies dir dort die Beschreibungen der Nodes durch
- oben in der Headerbar die **3 Punkte ...** öffnen ein Dropdown Menü, dort ist es möglich Workflows direkt zu importieren und exportieren
- links kann man zur Übersicht zurück navigieren

## 3.3 Workflow importieren
Wir haben einen Übungs-Workflow für dich vorbereitet.
Die Datei heißt `n8n_übung.json` und befindet sich im selben Verzeichnis wie dieses Notebook.

**Schritte:**
1. Rechts oben auf "Create Workflow" gehen
2. Danach auf "..." rechts oben und dann auf "Import from file..."
3. Navigiere in deinem Datei-Explorer zu dem Ordner, in dem dieses Notebook liegt, und wähle die Datei `n8n_übung_workflow.json` aus.

Nach dem Import solltest du den Workflow mit den verschiedenen Phasen (Host, Angry Customer, Scorer) sehen.